In [1]:
from libraries import *
from functions import *

import sankey
from importlib import reload
reload(sankey)
from sankey import *

In [2]:
adult_dataset= pd.read_csv("/Users/alessiabera/Downloads/adult.csv")
adult_dataset.head()
adult_dataset.shape


(32561, 15)

In [3]:
df_cs = adult_dataset.copy()

#x1=white
df_cs["X"] = (df_cs["race"] == "White").astype(int)

#y1=high
df_cs["Y"] = (df_cs["income"] == ">50K").astype(int)

#Grouping age (simpler)
df_cs["age_group"] = pd.cut(
    df_cs["age"],
    bins=[0, 25, 45, 65, 100],
    labels=["young", "mid", "senior", "elder"],
    include_lowest=True,
)


z_cols = ["age_group", "sex", "native.country"]      # confounders
w_cols = ["occupation", "relationship"]          # mediators


W_values = list(map(tuple, df_cs[w_cols].drop_duplicates().values))



Data preprocessing for readability

In [4]:
cols = ["X", "Y"] + z_cols + w_cols

df_cs = df_cs[
    ~(
        df_cs[cols].isna()        #NaN
       # | (df_cs[cols] == 0)      #numeric zero
        | (df_cs[cols] == "?")    #string "?"
    ).any(axis=1)
]

df_cs.shape

(30162, 18)

In [5]:
effects = compute_effects_multi(
    df=df_cs,
    x0=0,               #non-White
    x1=1,               #White
    y=1,                #income >50K
    W_values=W_values,  
    x_col="X",
    y_col="Y",
    w_cols=w_cols,
    z_cols=z_cols,
    do_decomposition=True
    
)

effects



{'tv': 0.08432624486858152,
 'te': 0.09492543058750987,
 'te_linear': 0.06764335121583097,
 'se': -0.010599185718928348,
 'se_x1': -0.0065178854688121395,
 'se_x0': 0.0040813002501162085,
 'de': 0.009128546210648333,
 'ie': -0.08579688437686155,
 'ie_f': 0.058514805005182635,
 'ie_decomp': {'occupation': 0.0, 'relationship': -0.08579688437686155},
 'se_decomp': {'age_group': -0.0016176959940954173,
  'sex': -0.02464959711401209,
  'native.country': 0.015668107389179158}}

Quick check

In [6]:

print("TV:", effects["tv"])
print("TE + SE:", effects["te"] + effects["se"])

print("SE:", effects["se"])
print("sum SE_decomp:", sum(effects["se_decomp"].values()))

print("IE forward:", effects["ie_f"])
print("sum IE_decomp:", sum(effects["ie_decomp"].values()))

print("IE_F+ DE:",effects["ie_f"]+ effects["de"])
print("TE:",effects["te"])
print("DE-IE:",effects["de"]-effects["ie"])


TV: 0.08432624486858152
TE + SE: 0.08432624486858152
SE: -0.010599185718928348
sum SE_decomp: -0.010599185718928348
IE forward: 0.058514805005182635
sum IE_decomp: -0.08579688437686155
IE_F+ DE: 0.06764335121583097
TE: 0.09492543058750987
DE-IE: 0.09492543058750988


In [7]:
plot_effect_sankey_percent(
    te=effects["te_linear"],          
    se=effects["se"],
    ie=effects["ie"],
    de=effects["de"],
    se_decomp=effects["se_decomp"],
    ie_decomp=effects["ie_decomp"],
    title="Adult: race → income – percent of |TV|"
)


x-specific effects

In [8]:
te_x, ie_x, de_x, se_x = compute_x_specific_effects(
    df=df_cs,
    x0=0,
    x1=1,
    x_cond=1,   #effect for individuals with X=1
    y_val=1,
    x_col="X",
    y_col="Y",
    w_cols=w_cols,      
    z_cols=z_cols  
)




print("X1-specific causal effects")
print(f"x-TE (ETT)      = {te_x:.6f}")
print(f"x-IE (Ctf-IE)   = {ie_x:.6f}")
print(f"x-DE (Ctf-DE)   = {de_x:.6f}")
print(f"x-SE (Ctf-SE)   = {se_x:.6f}")


te_x0, ie_x0, de_x0, se_x0 = compute_x_specific_effects(
    df=df_cs,
    x0=0,
    x1=1,
    x_cond=0,   #effect for individuals with X=0
    y_val=1,
    x_col="X",
    y_col="Y",
    w_cols=w_cols,      
    z_cols=z_cols  
)

print("X0-specific causal effects")
print(f"x-TE  = {te_x:.6f}")
print(f"x-IE = {ie_x:.6f}")
print(f"x-DE  = {de_x:.6f}")
print(f"x-SE= {se_x:.6f}")








X1-specific causal effects
x-TE (ETT)      = 0.100778
x-IE (Ctf-IE)   = -0.086197
x-DE (Ctf-DE)   = 0.014580
x-SE (Ctf-SE)   = -0.004747
X0-specific causal effects
x-TE  = 0.100778
x-IE = -0.086197
x-DE  = 0.014580
x-SE= -0.004747


In [9]:

plot_xspecific_sankey_percent(
    te_x=te_x,
    se_x=se_x,
    ie_x=ie_x,
    de_x=de_x,
    x_label="White",         
    exposure_name="Race",
    outcome_name="Income",
)




plot_xspecific_sankey_percent(
    te_x=te_x0,
    se_x=se_x0,
    ie_x=ie_x0,
    de_x=de_x0,
    x_label="Black",
    exposure_name="Race",
    outcome_name="Income",
)



z-specific effect

In [10]:
effects_z = compute_z_specific_effects(
    df=df_cs,
    x0=0,
    x1=1,
    y_val=1,
    x_col="X",
    y_col="Y",
    w_cols=w_cols,   
    z_cols=z_cols,  
)



#for z_tuple, effs in effects_z.items():
   # z_TE = effs["z_TE"]
   # z_DE = effs["z_DE"]
   # z_IE = effs["z_IE"]

   # if abs(z_TE) > 1e-12 or abs(z_DE) > 1e-12 or abs(z_IE) > 1e-12:
       # print(
        #    f"Z = {z_tuple}: "
        #    f"z-TE = {z_TE:.4f}, "
        #    f"z-DE = {z_DE:.4f}, "
        #    f"z-IE = {z_IE:.4f}"
      #  )


In [11]:
effects_z = compute_z_specific_effects(
    df=df_cs,
    x0=0,
    x1=1,
    y_val=1,
    x_col="X",
    y_col="Y",
    w_cols=w_cols,
    z_cols=z_cols,
)

plot_z_specific_decomposition(effects_z, top_k=20)


In [12]:
z_target = ("mid", "Male", "Cuba")
z_DE = effects_z[z_target]["z_DE"]
z_IE = effects_z[z_target]["z_IE"]

plot_z_specific_sankey(
    z_tuple=z_target,
    z_DE=z_DE,
    z_IE=z_IE,
    exposure_name="Race",
    outcome_name="Income",
)
